In [ ]:
import os
print("Current Working Directory:", os.getcwd())
print("Files and folders here:", os.listdir("."))


In [ ]:
label_dir_path = "../data/label"

if os.path.exists(label_dir_path):
    print("==============================================")
    print("         FILES INSIDE THE LABEL FOLDER        ")
    print("==============================================")
    # List the first 20 items inside the label directory
    items = os.listdir(label_dir_path)
    print(f"Total items in label folder: {len(items)}")
    print("First 20 items:", items[:20])
    print("==============================================")
else:
    print(f"Error: Could not find the folder at {label_dir_path}")

In [ ]:
label_file_path = "../data/label/label.csv"

if os.path.exists(label_file_path):
    # Load the master csv file
    label_df = pd.read_csv(label_file_path)
    
    print("==============================================")
    print("        SUCCESSFULLY LOADED MASTER LABELS     ")
    print("==============================================")
    print(f"Data Shape: {label_df.shape} (Rows, Columns)")
    print("\n--- Column Names ---")
    print(label_df.columns.tolist())
    print("\n--- First 3 Rows of the Dataset ---")
    print(label_df.head(3))
    print("==============================================")
else:
    print(f"Error: Could not find the file at {label_file_path}")

In [ ]:
import pickle
import os
import pandas as pd

data_file_path = "../data/data/0.pkl"

if os.path.exists(data_file_path):
    all_objects = []
    
    with open(data_file_path, 'rb') as f:
        while True:
            try:
                # Keep loading objects sequentially from the stream
                obj = pickle.load(f)
                all_objects.append(obj)
            except EOFError:
                # Stop smoothly when you reach the actual end of the file
                break
                
    print("==============================================")
    print("        STREAM UNPACKING COMPLETE             ")
    print("==============================================")
    print(f"Total separate objects found in file: {len(all_objects)}")
    
    # Let's inspect what each hidden layer is
    for idx, item in enumerate(all_objects):
        print(f"Layer {idx} -> Type: {type(item)}")
        if isinstance(item, pd.DataFrame) or hasattr(item, 'shape'):
            print(f"         Dimensions/Shape: {item.shape}")
        else:
            print(f"         Value: {item}")
    print("==============================================")
else:
    print(f"Error: Could not find the file at {data_file_path}")

In [ ]:
import pickle
import numpy as np
import pandas as pd

def extract_ev_features(file_path):
    with open(file_path, 'rb') as f:
        all_objects = []
        while True:
            try:
                all_objects.append(pickle.load(f))
            except EOFError:
                break
                
    # Grab the tuple from Layer 3
    data_tuple = all_objects[3]
    telemetry_matrix = data_tuple[0]  # The (128, 8) numpy array
    meta_dict = data_tuple[1]          # The dictionary with car, mileage, etc.
    
    # Calculate summary statistics across the 128 time steps for our 8 columns
    column_means = np.mean(telemetry_matrix, axis=0)  # Gives 8 average values
    column_maxes = np.max(telemetry_matrix, axis=0)  # Gives 8 maximum values
    column_mins = np.min(telemetry_matrix, axis=0)   # Gives 8 minimum values
    
    # Build a dictionary for a single clean row of data
    feature_row = {
        'car': meta_dict['car'],
        'mileage': meta_dict['mileage'],
        # Add the 8 aggregated telemetry columns (example mappings)
        'avg_cell_voltage': column_means[0],
        'max_cell_voltage': column_maxes[0],
        'min_cell_voltage': column_mins[0],
        'avg_current': column_means[1],
        'avg_temp': column_means[2],
        'max_temp': column_maxes[2],
        # Feature Engineering: Voltage unbalance gap
        'voltage_gap': column_maxes[0] - column_mins[0]
    }
    
    return feature_row

# Test it out on 0.pkl!
print(extract_ev_features("../data/data/0.pkl"))

In [ ]:
import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

data_folder = "../data/data"
all_files = sorted([f for f in os.listdir(data_folder) if f.endswith(".pkl")])
print(f"Found {len(all_files)} files. Utilizing thread pool concurrency to process...")

def process_single_file(file_name):
    file_path = os.path.join(data_folder, file_name)
    try:
        return extract_ev_features(file_path)
    except Exception:
        return None

# ThreadPoolExecutor works perfectly inside interactive notebooks on Windows!
# We set max_workers=16 to read 16 files simultaneously
with ThreadPoolExecutor(max_workers=16) as executor:
    results = list(executor.map(process_single_file, all_files))

# Filter out any files that failed or returned None
all_rows = [r for r in results if r is not None]

# Convert to DataFrame
features_df = pd.DataFrame(all_rows)

# Load master labels registry and merge
label_df = pd.read_csv("../data/label/label.csv")
final_training_dataset = pd.merge(features_df, label_df, on="car")

print("\n==============================================")
print("        CONCURRENT PROCESSING COMPLETE        ")
print("==============================================")
print(f"Total Combined Rows: {final_training_dataset.shape[0]}")
print(f"Unique Cars Processed: {final_training_dataset['car'].nunique()}")
print("==============================================")

# Save the full clean dataset
final_training_dataset.to_csv("../data/final_training_dataset.csv", index=False)

In [ ]:
import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

# Process all files concurrently
with ThreadPoolExecutor(max_workers=16) as executor:
    results = list(executor.map(process_single_file, all_files))

# 📋 TABLE 1: The Features Matrix (X)
# Pulls all the calculated sensor metrics from the pickle files
features_df = pd.DataFrame([r for r in results if r is not None])

# 📋 TABLE 2: The Labels Registry (y)
# Contains the clean ground-truth classifications
label_df = pd.read_csv("../data/label/label.csv")

# Optional: If you want to make sure both tables only contain the exact same cars 
# without permanently merging them into one giant spreadsheet:
valid_cars = features_df['car'].unique()
filtered_label_df = label_df[label_df['car'].isin(valid_cars)].copy()

print("\n==============================================")
print("        SEPARATE TABLES CREATED SUCCESSFULLY   ")
print("==============================================")
print(f"Table 1 (Features) Shape : {features_df.shape} -> Columns: {list(features_df.columns)}")
print(f"Table 2 (Labels) Shape   : {filtered_label_df.shape} -> Columns: {list(filtered_label_df.columns)}")
print("==============================================")

# Save them as two completely independent files
features_df.to_csv("../data/ev_features_X.csv", index=False)
filtered_label_df.to_csv("../data/ev_labels_y.csv", index=False)
print("Saved separate files: 'ev_features_X.csv' and 'ev_labels_y.csv'")